Data understanding and preprocessing


In [2]:
#Imports and Setup
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from geopy.distance import geodesic


In [3]:
#Load the dataset
file_path = '../Data/FinalDataset/merged_all.csv'
df = pd.read_csv(file_path, encoding='utf-8', low_memory=False)

print("Dataset Information:")
print(f"Total Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Information:
Total Rows: 5368
Columns: ['id', 'name', 'description', 'address', 'latitude', 'longitude', 'location', 'types']

Missing Values:
id               0
name             0
description      3
address        364
latitude         1
longitude        1
location        39
types            0
dtype: int64


In [43]:
#Data Cleaning
df.drop_duplicates(inplace=True)

columns_to_fill = {
        'name': 'Unknown Place',
        'description': 'No description available',
        'address': 'Address not provided',
        'location': 'Unspecified',
        'types': 'misc',
        'latitude': 0,
        'longitude': 0
    }
df.fillna(columns_to_fill, inplace=True)
print(df.isnull().sum())


id             0
name           0
description    0
address        0
latitude       0
longitude      0
location       0
types          0
dtype: int64


In [ ]:
#Encoding the types attribute
encoder = OneHotEncoder(sparse_output=False)
types_encoded = encoder.fit_transform(combined_df[['types']])
types_encoded_df = pd.DataFrame(types_encoded, columns=encoder.get_feature_names_out(['types']))
print("\nEncoded 'types' columns:")
print(types_encoded_df.head())

In [ ]:
#Merge encoded types back to the original dataframe 
combined_df = pd.concat([combined_df, types_encoded_df], axis=1)
print("\nDataFrame after merging encoded 'types':")
print(combined_df.head())

In [ ]:
#Normalize Latitude and Longitude
if 'latitude' in combined_df.columns and 'longitude' in combined_df.columns:
	scaler = MinMaxScaler()
	geo_features = combined_df[['latitude', 'longitude']]
	geo_normalized = scaler.fit_transform(geo_features)
	geo_normalized_df = pd.DataFrame(geo_normalized, columns=['latitude_norm', 'longitude_norm'])
	print("\nNormalized latitude and longitude:")
	print(geo_normalized_df.head())
else:
	print("\nColumns 'latitude' and 'longitude' are not present in the dataframe.")

In [ ]:
#Merge normalized geo features back to the original dataframe
combined_df = pd.concat([combined_df, geo_normalized_df], axis=1)
print("\nDataFrame after merging normalized geo features:")
print(combined_df.head())

In [8]:
processed_file_path = './processed_data.csv'
combined_df.to_csv(processed_file_path, index=False)

In [ ]:
print("Preprocessed DataFrame:")
print(combined_df.head())

Model Design


In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from geopy.distance import geodesic

In [ ]:
# Load the processed dataset
file_path = './processed_data.csv'
combined_df = pd.read_csv(file_path)

In [22]:
# User Input
def get_user_preferences():
    print("Select your preferred types from the following options:")
    unique_types = combined_df['types'].unique()
    for idx, t in enumerate(unique_types):
        print(f"{idx+1}. {t}")
    
    selected_types = input("Enter the numbers corresponding to your selected types, separated by commas (e.g., 1, 2, 3): ")
    selected_types = [int(x) - 1 for x in selected_types.split(',')]
    return unique_types[selected_types]

In [23]:
#Matching Mechanism - One-hot encoding
def match_user_preferences(user_selected_types):
    encoder = OneHotEncoder(sparse_output=False)
    types_encoded = encoder.fit_transform(combined_df[['types']])
    types_encoded_df = pd.DataFrame(types_encoded, columns=encoder.get_feature_names_out(['types']))

    # Create a binary vector for the user preferences
    user_vector = np.zeros(len(types_encoded_df.columns))
    for t in user_selected_types:
        idx = types_encoded_df.columns.str.contains(t)
        user_vector[idx] = 1

    #Compute Similarity Based on Types
    cosine_sim = cosine_similarity(user_vector.reshape(1, -1), types_encoded_df)
    return cosine_sim

In [26]:
# Optional Refinements - Geographic Proximity
def geographic_proximity(user_lat, user_lon, threshold_km=5):
    # Calculate distances from the user's location to all other places
    distances = combined_df.apply(
        lambda row: geodesic((user_lat, user_lon), (row['latitude'], row['longitude'])).km
        if pd.notna(row['latitude']) and pd.notna(row['longitude']) else float('inf'),
        axis=0
    )
    
    # Filter places that are within the distance threshold
    nearby_places = combined_df[distances <= threshold_km]
    return nearby_places

In [27]:
# Descriptive Similarity (TF-IDF & Cosine Similarity)
def descriptive_similarity(place_name):
    # Extract descriptions and compute TF-IDF
    tfidf = TfidfVectorizer(stop_words='english')
    descriptions = combined_df['description'].fillna('')
    tfidf_matrix = tfidf.fit_transform(descriptions)
    
    # Compute cosine similarity between the place and all other places
    place_idx = combined_df[combined_df['name'] == place_name].index[0]
    cosine_sim = cosine_similarity(tfidf_matrix[place_idx], tfidf_matrix)
    return cosine_sim.flatten()

In [28]:
#Filter Places by Location (same area)
def filter_by_location(places_df, location):
    return places_df[places_df['location'] == location]

In [29]:
# Combine Everything - Final Recommendation
def recommend_places(user_selected_types, user_lat, user_lon, threshold_km=5):
    # Filter places based on user-selected types (keep all types, no removal)
    filtered_places = combined_df[combined_df['types'].apply(
        lambda types: any(t.strip() in user_selected_types for t in types.split(',')) if isinstance(types, str) else False)]
    
    # Geographic proximity: Find places near user's location
    nearby_places = geographic_proximity(user_lat, user_lon, threshold_km)
    
    # Get places that match both user preferences (types) and proximity
    recommended_places = filtered_places[filtered_places['name'].isin(nearby_places['name'])]
    
    return recommended_places.head(10)

In [ ]:
user_selected_types = ['park', 'temple']  # User input for types (e.g., park or temple)
user_lat = 27.7172  # Example: User's latitude (can be current location or desired search location)
user_lon = 85.3240  # Example: User's longitude (can be current location or desired search location)

recommended_places = recommend_places(user_selected_types, user_lat, user_lon, threshold_km=5)

if not recommended_places.empty:
    print("Recommended Places based on user preferences and proximity:")
    print(recommended_places[['name', 'types', 'latitude', 'longitude']])
else:
    print("No recommendations found.")

In [1]:
import torch 
 
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")
print("Using CUDA" if cuda else "Using CPU")

Using CPU


In [2]:
import torch
print(torch.__version__)  # Check PyTorch version
print(torch.version.cuda) # Check the CUDA version PyTorch is using
print(torch.backends.cudnn.enabled)  # Check if cuDNN is enabled


2.5.1+cpu
None
True
